In [3]:
src_sent = '張三在台北101前面拍了很多照片。'

# Step1: 對原句作分詞

In [1]:
!pip install -qU pip setuptools wheel
!pip install -qU spacy
!python -m spacy download zh_core_web_sm
!python -m spacy download zh_core_web_md
!python -m spacy download zh_core_web_lg
!python -m spacy download zh_core_web_trf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('zh_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 MB 54.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('zh_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need 

In [4]:
import spacy
nlp = spacy.load("zh_core_web_trf")
doc = nlp(src_sent)

tok = list(list(token.text for token in doc))
print(tok)

ent = list(list(str(ent) for ent in doc.ents))
print(ent)

/usr/local/lib/python3.10/dist-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


['張三', '在', '台北', '101', '前面', '拍', '了', '很多', '照片', '。']
['張三', '台北101']


# Step2: 作翻譯

In [8]:
!pip install -q googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.7 MB/s eta 0:00:00


In [ ]:
from googletrans import Translator

# 翻譯器
translator = Translator()

current_translation = translator.translate(src_sent, src = "zh-TW", dest='en')
print(current_translation.text)

In [ ]:
target_sent = current_translation.text

# Step3: 對翻譯的句子作分詞

In [ ]:
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_md
!python -m spacy download en_core_web_lg
!python -m spacy download en_core_web_trf

In [ ]:
en_nlp = spacy.load("en_core_web_trf")
target_doc = en_nlp(target_sent)

target_tok = list(list(token.text for token in target_doc))
print(target_tok)

target_ent = list(list(str(ent) for ent in target_doc.ents))
print(target_ent)

#Step4: 對原句以及翻譯的句子重新分詞

### Step 4-0: 於中文語言中，初步探索加入自定義的分詞規則

ref: https://spacy.io/usage/rule-based-matching#entityruler

In [ ]:
import spacy
from spacy.tokens import Span
from spacy.pipeline import EntityRuler

# 載入中文語言模型
nlp = spacy.load("zh_core_web_trf")

# 定義自定義的分詞規則
ruler = nlp.add_pipe("entity_ruler")
patterns = [{"label": "SPECIAL_TERM", "pattern": label} for label in ent ]
ruler.add_patterns(patterns)

# 加入自訂處理邏輯，把「自然語言處理」當作一個整體字處理
@nlp.component("custom_tokenizer")
def custom_tokenizer(doc):
  # 使用 retokenizer 合併配對到的片語為一個 token
  with doc.retokenize() as retokenizer:
    for ent in doc.ents:
      retokenizer.merge(ent) # 合併實體為單一token

  return doc

# 把自訂分詞器加到 pipeline 裡
nlp.add_pipe("custom_tokenizer", last=True)

# 測試例子
doc = nlp("我愛自然語言處理")

# 看分詞結果
for token in doc:
  print(token.text)

我
爱
自然
语言
处理


## Step 4-1: 動態創建自定義 tokenizer [中文 ＆ 英文]

In [ ]:
src_dic =  {k: [str(k)] for k in ent}
print(src_dic)

{'張三': ['張三'], '台北101': ['台北101']}


In [ ]:
trg_dic =  {k: [str(k)] for k in target_ent}
print(trg_dic)

{'Zhang San': ['Zhang San'], 'Taipei 101': ['Taipei 101']}


In [ ]:
import spacy
from spacy.language import Language
from spacy.tokens import Doc

# 加載中文和英文語言模型
nlp_zh = spacy.load("zh_core_web_trf")
nlp_en = spacy.load("en_core_web_trf")

nlp_zh.add_pipe("entity_ruler", before="ner")
nlp_en.add_pipe("entity_ruler", before="ner")

# 動態創建中文自定義 tokenizer 的工廠函數
@Language.factory("custom_tokenizer_zh")
def create_custom_tokenizer_zh(nlp, name, labels):
    def custom_tokenizer_zh(doc):
        patterns = [{"label": "SPECIAL_TERM", "pattern": label} for label in labels]
        ruler = nlp.get_pipe("entity_ruler")
        ruler.add_patterns(patterns)

        with doc.retokenize() as retokenizer:
            for ent in doc.ents:
                retokenizer.merge(ent)
        return doc
    return custom_tokenizer_zh

# 動態創建英文自定義 tokenizer 的工廠函數
@Language.factory("custom_tokenizer_en")
def create_custom_tokenizer_en(nlp, name, labels):
    def custom_tokenizer_en(doc):
        patterns = [{"label": "SPECIAL_TERM", "pattern": label} for label in labels]
        ruler = nlp.get_pipe("entity_ruler")
        ruler.add_patterns(patterns)

        with doc.retokenize() as retokenizer:
            for ent in doc.ents:
                retokenizer.merge(ent)
        return doc
    return custom_tokenizer_en

# 在中文和英文模型中添加自定義 tokenizer 到 pipeline
nlp_zh.add_pipe("custom_tokenizer_zh", last=True, config={"labels": src_dic})
nlp_en.add_pipe("custom_tokenizer_en", last=True, config={"labels": trg_dic})

ValueError: [E004] Can't set up pipeline component: a factory for 'custom_tokenizer_zh' already exists. Existing factory: <function create_custom_tokenizer_zh at 0x7a82529dbd90>. New factory: <function create_custom_tokenizer_zh at 0x7a8204ba9000>

In [ ]:
# 處理中文文本
doc_zh = nlp_zh(src_sent)
print("中文分詞結果:")
tok_zh = [token.text for token in doc_zh]
print(tok_zh)


# 處理英文文本
doc_en = nlp_en(target_sent)
print("\n英文分詞結果:")
tok_en = [token.text for token in doc_en]
print(tok_en)


/usr/local/lib/python3.10/dist-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/usr/local/lib/python3.10/dist-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


中文分詞結果:
['張三', '在', '台北101', '前面', '拍', '了', '很多', '照片', '。']

英文分詞結果:
['Zhang San', 'took', 'a', 'lot', 'of', 'photos', 'in', 'front', 'of', 'Taipei 101', '.']


/usr/local/lib/python3.10/dist-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


# Step5: 修正翻譯

先不使用套件，先用手動。


In [ ]:
term_alignment = {'張三': 'Zhang San', '台北101': 'Taipei 101'}

In [ ]:
corrected_sentence = tok_en.copy()

for k, v in term_alignment.items():
  if v in tok_en:
    corrected_sentence[tok_en.index(v)] = k

ret = " ".join(corrected_sentence[:-1])+"."
ret

'張三 took a lot of photos in front of 台北101.'

# 30 個繁體中文句子

* Generated by Chatgpt-4o.
* Ref with [a messy dialogue](https://chatgpt.com/share/d5b671c8-432d-4465-a8bc-5255c73b3ee6).

* **小結：好像有點難修改**

In [6]:
test_sentence = {
  "sentences": [
    "張三在台北101前面拍了很多照片。",
    "李四昨天在台中火車站迷路了。",
    "王五參加了鴻海公司的年度大會。",
    "華為手機在全球市場上的銷量持續增長。",
    "張三開了一間叫做「香格里拉」的餐廳。",
    "小明帶著他的iPhone去了華山藝文中心。",
    "你知道Apple的總部在哪裡嗎？",
    "昨天在星巴克遇見了老朋友林小明。",
    "上海是中國的經濟中心。",
    "Google正在開發一個新的AI模型。",
    "馬雲創辦的阿里巴巴是中國最大的電商平台之一。",
    "珠穆朗瑪峰是世界最高的山。",
    "高雄的六合夜市是美食愛好者的天堂。",
    "臺灣大學的學生來參加這次比賽。",
    "蘋果電腦的發明改變了整個科技產業。",
    "中國移動的用戶數量每年都在增加。",
    "香港的金融市場非常活躍。",
    "IBM推出了一款全新的量子計算機。",
    "Microsoft的Windows系統是全球最常用的操作系統之一。",
    "劉德華出演了很多經典的電影。",
    "台灣的玉山是當地最高的山峰。",
    "世界衛生組織在日內瓦總部召開會議。",
    "任天堂的Switch遊戲機在全球大受歡迎。",
    "中央電視台播放了關於新冠疫情的最新報導。",
    "奧林匹克運動會將在東京舉行。",
    "特斯拉的自動駕駛技術引發了廣泛關注。",
    "小紅書是一個非常受歡迎的社交平台。",
    "Facebook的隱私政策再次引發爭議。",
    "日本的富士山每年都吸引大量遊客。",
    "李小龍的武術精神影響了全世界。"
  ]
}

In [5]:
import spacy

doc = nlp(src_sent)

tok = list(list(token.text for token in doc))
print(tok)

ent = list(list(str(ent) for ent in doc.ents))
print(ent)

['張三', '在', '台北', '101', '前面', '拍', '了', '很多', '照片', '。']
['張三', '台北101']


In [9]:
from googletrans import Translator

nlp = spacy.load("zh_core_web_trf")


src_sent = []
trans_res = []
src_ents = []
# 翻譯器
translator = Translator()

for source_sentence in test_sentence["sentences"]:
  src_sent.append(source_sentence)
  current_translation = translator.translate(source_sentence, src = "zh-TW", dest='en')
  trans_res.append(current_translation.text)

  doc = nlp(source_sentence)
  ent = tuple(list(str(ent) for ent in doc.ents))
  src_ents.append(ent)




In [10]:
import pandas as pd

df = pd.DataFrame(list(zip(src_sent, src_ents, trans_res)),
               columns =['原句', '原句的專有名詞','翻譯'])

df.head(30)

,原句,原句的專有名詞,翻譯
0,張三在台北101前面拍了很多照片。,"(張三, 台北101)",Zhang San took a lot of photos in front of Tai...
1,李四昨天在台中火車站迷路了。,"(李四, 昨天, 台中火車站)",Li Si was lost at Taichung Railway Station yes...
2,王五參加了鴻海公司的年度大會。,"(王五參, 鴻海公司)",Wang Wu participated in the annual conference ...
3,華為手機在全球市場上的銷量持續增長。,(),The sales of Huawei mobile phones in the globa...
4,張三開了一間叫做「香格里拉」的餐廳。,"(張三, 一, 香格里拉)","Zhang San opened a restaurant called ""Shangri ..."
5,小明帶著他的iPhone去了華山藝文中心。,"(華山藝文中心,)",Xiaoming took his iPhone to Huashan Art and Cu...
6,你知道Apple的總部在哪裡嗎？,"(Apple,)",Do you know where the headquarters of Apple is?
7,昨天在星巴克遇見了老朋友林小明。,"(星巴克, 林小明)",I met an old friend Lin Xiaoming in Starbucks ...
8,上海是中國的經濟中心。,"(上海,)",Shanghai is the economic center of China.
9,Google正在開發一個新的AI模型。,"(Google,)",Google is developing a new AI model.
